# Task 4: Improved LCS-Based System

## Purpose

This notebook develops an improved eLCS-based diabetes classification system and compares it with the original eLCS baseline.

The improved system introduces three main changes:

1. Random undersampling is used to balance the training classes.
2. Predictor types are explicitly configured for eLCS so that binary variables are treated as discrete attributes, while ordered and numerical variables can form range-based rules.
3. A small hyperparameter search is performed using a separate validation set.

The original eLCS source code is not modified. All improvements are made through the machine-learning pipeline and the parameters provided by the original implementation.

The final test set remains unchanged and is not used during model selection. This prevents data leakage and allows a fair comparison with the original baseline.


In [2]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from skeLCS import eLCS

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Python version:", sys.version)
print("eLCS imported successfully")

Python version: 3.9.13 (tags/v3.9.13:6de2ca5, May 17 2022, 16:36:42) [MSC v.1929 64 bit (AMD64)]
eLCS imported successfully


## 1. Load the prepared training and test data

Notebook 05 produced a balanced training dataset using random undersampling and an unchanged test dataset. The balanced dataset contains equal numbers of both target classes, while the test set retains the original population distribution.

The unchanged test set is reserved for final evaluation and will not be used to select eLCS parameters.


In [3]:
data_directory = Path("../data/processed/elcs")

training_path = (
    data_directory
    / "diabetes_elcs_training_balanced.csv"
)

test_path = (
    data_directory
    / "diabetes_elcs_test_unchanged.csv"
)

training_data = pd.read_csv(training_path)
test_data = pd.read_csv(test_path)

print("Balanced training shape:", training_data.shape)
print("Unchanged test shape:", test_data.shape)

print("\nTraining class counts:")
print(
    training_data["Diabetes_binary"]
    .value_counts()
    .sort_index()
)

print("\nTest class counts:")
print(
    test_data["Diabetes_binary"]
    .value_counts()
    .sort_index()
)

print("\nTraining missing values:")
print(training_data.isna().sum().sum())

print("Test missing values:")
print(test_data.isna().sum().sum())

Balanced training shape: (56554, 22)
Unchanged test shape: (50736, 22)

Training class counts:
Diabetes_binary
0    28277
1    28277
Name: count, dtype: int64

Test class counts:
Diabetes_binary
0    43667
1     7069
Name: count, dtype: int64

Training missing values:
0
Test missing values:
0


## Validation split

The balanced training dataset was divided into a tuning-training set and a validation set using stratified sampling. The validation set will be used to compare eLCS configurations without using the final test data. The unchanged test set is reserved for the final comparison with the original baseline, preventing data leakage.

In [4]:
from sklearn.model_selection import train_test_split

target_column = "Diabetes_binary"

# Separate features and target
X_balanced = training_data.drop(columns=[target_column])
y_balanced = training_data[target_column].astype(int)

X_test = test_data.drop(columns=[target_column])
y_test = test_data[target_column].astype(int)

# Confirm that both datasets use the same features
assert list(X_balanced.columns) == list(X_test.columns)

# Create training and validation sets
X_tune_train, X_validation, y_tune_train, y_validation = (
    train_test_split(
        X_balanced,
        y_balanced,
        test_size=0.20,
        random_state=42,
        stratify=y_balanced
    )
)

print("Tuning training features:", X_tune_train.shape)
print("Validation features:", X_validation.shape)
print("Final test features:", X_test.shape)

print("\nTuning training percentages:")
print(y_tune_train.value_counts(normalize=True).mul(100).round(2))

print("\nValidation percentages:")
print(y_validation.value_counts(normalize=True).mul(100).round(2))

Tuning training features: (45243, 21)
Validation features: (11311, 21)
Final test features: (50736, 21)

Tuning training percentages:
Diabetes_binary
1    50.0
0    50.0
Name: proportion, dtype: float64

Validation percentages:
Diabetes_binary
0    50.0
1    50.0
Name: proportion, dtype: float64


## Attribute configuration

The eLCS system must distinguish between discrete categorical features and numerical or ordered features. Binary variables are treated as discrete categories. BMI, health-day counts and ordered variables are treated as range-based attributes so that eLCS can construct rules covering meaningful intervals. This is an improvement over relying entirely on automatic attribute detection.

In [5]:
import numpy as np

# Features that eLCS should represent using numerical ranges
range_based_features = [
    "BMI",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "Age",
    "Education",
    "Income"
]

# Convert column names into their numerical positions
specified_attributes = np.array(
    [
        X_tune_train.columns.get_loc(column)
        for column in range_based_features
    ],
    dtype=int
)

# All remaining features will be treated as discrete
discrete_features = [
    column
    for column in X_tune_train.columns
    if column not in range_based_features
]

print("Range-based features:")
for column, index in zip(
    range_based_features,
    specified_attributes
):
    print(index, "->", column)

print("\nDiscrete features:")
for column in discrete_features:
    print(X_tune_train.columns.get_loc(column), "->", column)

print("\nNumber of range-based features:", len(range_based_features))
print("Number of discrete features:", len(discrete_features))

Range-based features:
3 -> BMI
13 -> GenHlth
14 -> MentHlth
15 -> PhysHlth
18 -> Age
19 -> Education
20 -> Income

Discrete features:
0 -> HighBP
1 -> HighChol
2 -> CholCheck
4 -> Smoker
5 -> Stroke
6 -> HeartDiseaseorAttack
7 -> PhysActivity
8 -> Fruits
9 -> Veggies
10 -> HvyAlcoholConsump
11 -> AnyHealthcare
12 -> NoDocbcCost
16 -> DiffWalk
17 -> Sex

Number of range-based features: 7
Number of discrete features: 14


In [6]:
discrete_attribute_limit="c"
specified_attributes=specified_attributes